# Exploratory genre-corpus analysis

This notebook provides corpus statistics and a separate 70/10/20 splitting utility, not genre-classifier training. **It is optional and does not define the paper's training/evaluation protocol.** Use `Genre_Router_Training.ipynb` for the router.

Run the splitting utility before the analysis. Generated files are saved separately under `ENSEMBLE_OUTPUT`. Check inferred label columns before using these diagnostics because automatic binary-column inference can also include non-genre columns. The exploratory analysis is not part of the reported cross-domain evaluation.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

repo = Path("/content/genre-expert-ensembles") if Path("/content").exists() else Path.cwd()
if not (repo / "common.py").is_file():
    if not Path("/content").exists():
        raise RuntimeError("Run this notebook from the cloned repository root.")
    subprocess.run([
        "git", "clone",
        "https://github.com/Jun-Wang0/Domain-level-learning-and-genre-specific-expert-ensembles.git",
        str(repo),
    ], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

os.environ["ENSEMBLE_ROOT"] = "/content/drive/MyDrive/PhD"
os.environ["ENSEMBLE_OUTPUT"] = str(Path(os.environ["ENSEMBLE_ROOT"]) / "genre_ensemble_runs")
print("Repository:", repo)
print("New experiment outputs:", os.environ["ENSEMBLE_OUTPUT"])


In [ ]:
RUN_EXPLORATORY_ANALYSIS = False
if not RUN_EXPLORATORY_ANALYSIS:
    raise RuntimeError("Optional analysis is disabled. Read the notes before enabling it.")
for name in ["exploratory_splits", "exploratory_analysis"]:
    if (Path(os.environ["ENSEMBLE_OUTPUT"]) / name).exists():
        raise FileExistsError("Exploratory output already exists. Choose a new ENSEMBLE_OUTPUT.")

## Exploratory splitting utility

This creates a separate exploratory copy of the data. It does not modify the full-domain sentiment CSVs.

In [ ]:
"""
Step 0: freeze the train/dev/test partitions.
"""

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

SUBSAMPLE_N = 20000  # training subsample used for the cheap hyper-parameter grids

def infer_label_cols(df, text_col, group_col):
    """A label column is binary and is neither the text nor the id column."""
    reserved = {text_col, group_col, "Unnamed: 0", "index"}
    cols = []
    for c in df.columns:
        if c in reserved:
            continue
        s = df[c].dropna()
        if len(s) and s.isin([0, 1, 0.0, 1.0, True, False]).all():
            cols.append(c)
    return cols

def report_label_shift(full, subset, label_cols, name):
    """Random subsampling can quietly flatten rare genres. Report the drift."""
    p_full = full[label_cols].mean()
    p_sub = subset[label_cols].mean()
    drift = (p_sub - p_full).abs() / p_full.clip(lower=1e-9)
    worst = drift.idxmax()
    print(
        f"    [{name}] max relative label-frequency drift = {drift.max():.1%} "
        f"on '{worst}' ({p_full[worst]:.4f} -> {p_sub[worst]:.4f})"
    )
    return {"max_relative_drift": float(drift.max()), "worst_label": str(worst)}

def split_domain(csv, domain, out_root, text_col, group_col, seed):
    print(f"\n{'=' * 70}\n{domain}\n{'=' * 70}")
    df = pd.read_csv(csv)
    n_raw = len(df)

    label_cols = infer_label_cols(df, text_col, group_col)
    if not label_cols:
        raise ValueError(f"no binary label columns found in {csv}")

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].str.len() > 0]
    n_empty = n_raw - len(df)

    n_before = len(df)
    df = df.drop_duplicates(subset=[text_col]).reset_index(drop=True)
    n_dup = n_before - len(df)

    has_groups = group_col in df.columns and df[group_col].notna().all()
    if has_groups:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
        idx_tr, idx_te = next(gss.split(df, groups=df[group_col]))
        tr_all, te = df.iloc[idx_tr].reset_index(drop=True), df.iloc[idx_te].reset_index(drop=True)
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=seed)
        idx_tr2, idx_dv = next(gss2.split(tr_all, groups=tr_all[group_col]))
        tr, dv = tr_all.iloc[idx_tr2].reset_index(drop=True), tr_all.iloc[idx_dv].reset_index(drop=True)
    else:
        tr_all, te = train_test_split(df, test_size=0.20, random_state=seed)
        tr, dv = train_test_split(tr_all, test_size=0.125, random_state=seed)
        tr, dv, te = (x.reset_index(drop=True) for x in (tr, dv, te))

    rng = np.random.RandomState(seed)
    take = min(SUBSAMPLE_N, len(tr))
    sub = tr.iloc[rng.choice(len(tr), take, replace=False)].reset_index(drop=True)
    drift = report_label_shift(tr, sub, label_cols, f"train_{take // 1000}k")

    dom_dir = Path(out_root)
    dom_dir.mkdir(parents=True, exist_ok=True)
    for name, part in [("train", tr), ("dev", dv), ("test", te), (f"train_{take // 1000}k", sub)]:
        part.to_csv(dom_dir / f"{name}.csv", index=False)

    manifest = {
        "domain": domain,
        "source_csv": str(csv),
        "label_cols": label_cols,
        "n_train": len(tr), "n_dev": len(dv), "n_test": len(te),
        "test_fingerprint": hashlib.md5("".join(sorted(te[text_col].head(1000))).encode()).hexdigest()[:12]
    }
    with open(dom_dir / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"  Written to {dom_dir}")
    return manifest

# --- Configured Task List ---
tasks = [
    {
        "domain": "Book",
        "file_path": str(Path(os.environ["ENSEMBLE_ROOT"]) / "DataSet/Book/merged_balanced_reviews_multi_genre.csv"),
        "output_dir": str(Path(os.environ["ENSEMBLE_OUTPUT"]) / "exploratory_splits" / "Book")
    },
    {
        "domain": "Movie",
        "file_path": str(Path(os.environ["ENSEMBLE_ROOT"]) / "DataSet/Movie/merged_balanced_reviews_multi_genre.csv"),
        "output_dir": str(Path(os.environ["ENSEMBLE_OUTPUT"]) / "exploratory_splits" / "Movie")
    },
    {
        "domain": "Game",
        "file_path": str(Path(os.environ["ENSEMBLE_ROOT"]) / "DataSet/Game/merged_balanced_reviews_multi_genre.csv"),
        "output_dir": str(Path(os.environ["ENSEMBLE_OUTPUT"]) / "exploratory_splits" / "Game")
    }
]

# --- Execution ---
manifests = []
for t in tasks:
    m = split_domain(
        csv=t["file_path"],
        domain=t["domain"],
        out_root=t["output_dir"],
        text_col="review",
        group_col="item_id",
        seed=42
    )
    manifests.append(m)

print("\nAll splits frozen.")
for m in manifests:
    print(f"  {m['domain']}: {m['test_fingerprint']}")

## Original corpus analysis

In [ ]:
"""
A-group analysis.

A1  Corpus statistics       -> Table 1 and length distributions
A2  Label co-occurrence     -> Conditional P(j|i) and Jaccard
A3  Genre separability      -> TF-IDF centroid distances
"""

import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

# --- Paths Configuration (Linked to your frozen data) ---
DATA_ROOT = str(Path(os.environ["ENSEMBLE_OUTPUT"]) / "exploratory_splits")
OUT_DIR = str(Path(os.environ["ENSEMBLE_OUTPUT"]) / "exploratory_analysis")
DOMAINS = ["Book", "Movie", "Game"]
TOKENIZER_NAME = "roberta-large"
TRUNC_POINTS = [128, 256, 512]

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def load_domain(data_root, domain):
    d = Path(data_root) / domain
    man = json.load(open(d / "manifest.json"))
    parts = {k: pd.read_csv(d / f"{k}.csv") for k in ["train", "dev", "test"]}
    parts["full"] = pd.concat(parts.values(), ignore_index=True)
    return man, parts

def latex_table(df, caption, label, float_fmt="%.2f"):
    body = df.to_latex(index=False, escape=True, float_format=lambda x: float_fmt % x,
                       column_format="l" + "r" * (df.shape[1] - 1))
    body = body.replace("\\toprule", "\\hline").replace("\\midrule", "\\hline").replace("\\bottomrule", "\\hline")
    return (f"\\begin{{table}}[t]\n\\centering\n\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n{body}\\end{{table}}\n")

# ---------------------------------------------------------------------------
# A1: corpus statistics
# ---------------------------------------------------------------------------
def a1_corpus_stats(domain, man, parts, tokenizer, out_dir):
    text_col = "review"
    labels = man["label_cols"]
    full = parts["full"]
    texts = full[text_col].astype(str).tolist()

    # Word counts
    words = np.array([len(t.split()) for t in texts])

    # Tokenization lengths
    tok_len = []
    print(f"  [{domain}] Tokenizing reviews...")
    for i in range(0, len(texts), 2000):
        enc = tokenizer(texts[i:i + 2000], add_special_tokens=True, truncation=False)
        tok_len.extend(len(x) for x in enc["input_ids"])
    tok_len = np.array(tok_len)

    row = {
        "Domain": domain,
        "Reviews": len(full),
        "Words (med)": float(np.median(words)),
        "Words (p90)": float(np.percentile(words, 90)),
        "Tokens (med)": float(np.median(tok_len)),
        "Tokens (p90)": float(np.percentile(tok_len, 90)),
        "Labels/review": float(full[labels].sum(axis=1).mean()),
    }
    for L in TRUNC_POINTS:
        row[f"Trunc@{L} (%)"] = float((tok_len > L).mean() * 100)

    np.save(out_dir / f"a1_{domain}_token_lengths.npy", tok_len)
    return row, tok_len

def a1_length_figure(all_lengths, out_dir):
    fig, ax = plt.subplots(figsize=(7, 4))
    for domain, tl in all_lengths.items():
        x = np.sort(tl)
        ax.plot(x, np.arange(1, len(x) + 1) / len(x), label=domain, lw=2)
    ax.set_xscale("log")
    ax.set_xlim(10, 3000)
    ax.set_xlabel("RoBERTa tokens (log scale)")
    ax.set_ylabel("Cumulative fraction")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / "fig1_length_cdf.png", dpi=300)
    plt.show()

# ---------------------------------------------------------------------------
# A2 & A3: Co-occurrence and Separability
# ---------------------------------------------------------------------------
def a2_a3_analysis(domain, man, parts, out_dir):
    labels = man["label_cols"]
    Y = parts["full"][labels].values.astype(float)

    # A2: Co-occurrence
    joint = Y.T @ Y
    marg = Y.sum(axis=0)
    cond = joint / np.maximum(marg[:, None], 1e-9)
    pd.DataFrame(cond, index=labels, columns=labels).to_csv(out_dir / f"a2_{domain}_cond_full.csv")

    # A3: Separability via TF-IDF Centroids
    tr = parts["train"]
    tfidf = TfidfVectorizer(max_features=15000, stop_words="english", sublinear_tf=True)
    X = tfidf.fit_transform(tr["review"].astype(str))
    Y_tr = tr[labels].values

    centroids = []
    for k in range(len(labels)):
        if Y_tr[:, k].sum() > 0:
            centroids.append(np.asarray(X[Y_tr[:, k] == 1].mean(axis=0)).ravel())
        else:
            centroids.append(np.zeros(X.shape[1]))

    sim = cosine_similarity(np.vstack(centroids))
    off_diag = ~np.eye(len(labels), dtype=bool)

    return {"Domain": domain, "MeanCentroidDist": float(1 - sim[off_diag].mean())}

# --- Main Execution ---
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

a1_rows, a3_rows, lengths = [], [], {}

for d in DOMAINS:
    print(f"\n--- Processing Domain: {d} ---")
    man, parts = load_domain(DATA_ROOT, d)

    # Stats
    row, tl = a1_corpus_stats(d, man, parts, tok, out)
    a1_rows.append(row)
    lengths[d] = tl

    # Separability
    a3_rows.append(a2_a3_analysis(d, man, parts, out))

# Visualize and Export
a1_length_figure(lengths, out)

t1 = pd.DataFrame(a1_rows)
t3 = pd.DataFrame(a3_rows)

t1.to_csv(out / "table1_corpus_stats.csv", index=False)
with open(out / "table1.tex", "w") as f:
    f.write(latex_table(t1, "Corpus statistics and truncation rates", "tab:stats"))

print("\n--- Analysis Summary ---")
display(t1)
print("\n--- Genre Separability Summary ---")
display(t3)